
# Time-Series Forecasting with Feature-Based ML

## Why time-series forecasting is different

Time-series problems require a different workflow than standard tabular ML:

-   Observations are *ordered in time*, so shuffling usually causes leakage.
-   Most useful predictors are *past values* (lags, rolling statistics, seasonal patterns).
-   Evaluation should mimic deployment with *future-only* validation.

Guiding question: **Can we outperform strong naive baselines with feature-based ML while avoiding leakage?**

### Forecast framing terminology

In forecasting, three terms are essential:

-   **Forecast origin**: the last timestamp you are allowed to use for training.
-   **Horizon**: how far into the future you predict (1-step, 6-step, etc.).
-   **Window**: the historical context used to build features.

For this notebook, we use a 1-step horizon and evaluate repeatedly at new forecast origins.

### Leakage checklist (use this in every project)

Before trusting any metric, verify:

1.  No random shuffle across time.
2.  No feature uses future values (directly or indirectly).
3.  Preprocessing statistics (if any) are computed on train data only.
4.  Hyperparameter decisions are made on validation, not test.
5.  Final test is run once, after model selection.

### Mathematical problem statement

Let $\{y_t\}_{t=1}^{T}$ be a univariate time series. For one-step forecasting, at time $t$ we predict: $$ \hat{y}_{t+1|t} = f_{\theta}(\phi_t) $$ where:

-   $\phi_t$ is a feature vector built only from information available at time $t$,
-   $f_{\theta}$ is the model with parameters $\theta$.

Training solves empirical risk minimization: $$ \hat{\theta} = \arg\min_{\theta}\frac{1}{N}\sum_{t \in \mathcal{T}_{train}} \mathcal{L}\left(y_{t+1}, f_{\theta}(\phi_t)\right) $$

Important interpretation:

-   with squared loss, the optimum approximates $E[y_{t+1} \mid \phi_t]$,
-   with absolute loss, the optimum approximates the conditional median.

So feature engineering is not "extra"; it defines what conditional signal the model can learn.

## Running example: Monthly atmospheric CO2

We will use the classic Mauna Loa CO2 dataset from `statsmodels`. This is a useful teaching dataset because it clearly shows long-term trend and seasonal structure.

### Signal decomposition intuition

A common conceptual decomposition is: $$ y_t = T_t + S_t + R_t $$ where:

-   $T_t$: trend (slow long-term movement),
-   $S_t$: seasonal component (repeating calendar pattern),
-   $R_t$: residual/noise component.

Feature-based ML does not explicitly estimate these components, but lag and calendar features try to capture them implicitly.

### Load and visualize the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.datasets import co2

sns.set_theme(style="whitegrid")
np.random.seed(42)

# Weekly CO2 -> monthly start frequency
y = co2.load_pandas().data["co2"].resample("MS").mean().interpolate()
y.name = "co2"

print(f"Range: {y.index.min().date()} to {y.index.max().date()} | n={len(y)}")

ax = y.plot(figsize=(11, 4), lw=2, title="Monthly Atmospheric CO2")
ax.set_ylabel("ppm")
plt.show()

**Interpretation**:

-   The long-term upward slope indicates trend.
-   Repeating within-year oscillation suggests seasonality.
-   A "last value only" model may already do reasonably well, which is why baseline comparison is mandatory.

### Quick autocorrelation check

If lagged values are informative, sample autocorrelation at those lags should be non-trivial.

In [ ]:
for lag in [1, 2, 3, 6, 12, 24]:
    print(f"lag={lag:>2} autocorr={y.autocorr(lag=lag):.3f}")

High autocorrelation at lag 12 is a strong hint that yearly seasonality features are useful.

### Train/validation/test split (time-aware)

We split by date, not randomly:

-   Train: 1958-03 to 1989-12
-   Validation: 1990-01 to 1995-12
-   Test: 1996-01 to 2001-12

In [ ]:
train_end = "1989-12-01"
val_end = "1995-12-01"
test_start = "1996-01-01"

y_train = y.loc[:train_end]
y_val = y.loc["1990-01-01":val_end]
y_test = y.loc[test_start:]

print("Train:", y_train.index.min().date(), "->", y_train.index.max().date(), f"(n={len(y_train)})")
print("Val:  ", y_val.index.min().date(), "->", y_val.index.max().date(), f"(n={len(y_val)})")
print("Test: ", y_test.index.min().date(), "->", y_test.index.max().date(), f"(n={len(y_test)})")

**Why this split design?**

-   Training block is long enough to learn multiple seasonal cycles.
-   Validation block is contiguous in time, so it reflects genuine forward prediction.
-   Test block is held out for a single final comparison after model choices are fixed.

If a model does great on validation and fails on test, that usually indicates:

-   over-tuning on validation
-   regime shift (data-generating process changed)
-   hidden leakage

### Why chronological splits are statistically coherent

Forecasting assumes an information flow: $$ \mathcal{F}_{t_1} \subset \mathcal{F}_{t_2} \quad \text{for } t_1 < t_2 $$ where $\mathcal{F}_t$ is the information set available at time $t$. Random shuffling violates this temporal filtration by mixing "future-conditioned" and "past-conditioned" samples in train and test.

## Baselines first: naive and seasonal naive

Before training any model, create strong baselines:

-   **Naive (t-1)**: predict next month as last month.
-   **Seasonal naive (t-12)**: predict next month as same month last year.

In notation:

-   $\hat{y}_{t+1}^{naive} = y_t$
-   $\hat{y}_{t+1}^{seasonal} = y_{t+1-12}$ for monthly data

Seasonal naive is often surprisingly strong for environmental, retail, and energy data. If your ML model cannot beat it consistently, the feature/model complexity is not justified.

### When these baselines are theoretically strong

-   If $y_t$ follows a random walk $y_t = y_{t-1} + \varepsilon_t$, then naive forecast is optimal under squared loss.
-   If $y_t$ follows a seasonal random walk $y_t = y_{t-s} + \varepsilon_t$, then seasonal naive is optimal.

This is why baselines are not "toy" methods. They are implied by explicit stochastic assumptions.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def forecast_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {"MAE": mae, "RMSE": rmse, "MAPE(%)": mape}

naive_pred = y.shift(1).loc[y_test.index]
seasonal_naive_pred = y.shift(12).loc[y_test.index]

baseline_results = pd.DataFrame({
    "Naive (t-1)": forecast_metrics(y_test, naive_pred),
    "Seasonal Naive (t-12)": forecast_metrics(y_test, seasonal_naive_pred),
}).T

baseline_results.round(3)

### Why these metrics?

-   **MAE**: average absolute error in original units; robust and easy to explain.
-   **RMSE**: penalizes large misses more strongly; useful when big errors are costly.
-   **MAPE**: percentage error; intuitive but can be unstable when true values are near zero.

For CO2 (values far from zero), MAPE is stable enough for comparison.

In formulas, with errors $e_i = y_i - \hat{y}_i$: $$ MAE = \frac{1}{n}\sum_{i=1}^n |e_i|,\quad RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^n e_i^2},\quad MAPE = \frac{100}{n}\sum_{i=1}^n \left|\frac{e_i}{y_i}\right| $$

Metric choice is a business/research decision:

-   choose MAE when typical absolute error matters,
-   choose RMSE when large misses are disproportionately costly,
-   report both when stakes are mixed.

## Feature engineering for time-series ML

We convert the series into a supervised table using:

-   lag features
-   rolling means/std (computed on *past-only* values)
-   cyclical month features (sin/cos)
-   simple trend index

Conceptually, we are transforming:

-   input: one column $y_t$ over time
-   into: many explanatory columns at time $t$ that are known at or before $t$

The model then learns a mapping $f(X_t) \to y_t$, where $X_t$ contains history-derived features.

More explicitly, for monthly data: $$ \phi_t = \left[ y_{t-1}, y_{t-2}, y_{t-3}, y_{t-6}, y_{t-12}, \mu_{t-1}^{(3)}, \mu_{t-1}^{(6)}, \mu_{t-1}^{(12)}, \sigma_{t-1}^{(3)}, \sigma_{t-1}^{(6)}, \sigma_{t-1}^{(12)}, \sin\left(\frac{2\pi m_t}{12}\right), \cos\left(\frac{2\pi m_t}{12}\right), t \right] $$ where: $$ \mu_{t-1}^{(w)} = \frac{1}{w}\sum_{j=1}^{w} y_{t-j} $$ and $\sigma_{t-1}^{(w)}$ is the corresponding rolling standard deviation.

In [ ]:
def make_feature_table(series, lags=(1, 2, 3, 6, 12), rolling_windows=(3, 6, 12)):
    df = pd.DataFrame({"y": series})

    for lag in lags:
        df[f"lag_{lag}"] = series.shift(lag)

    shifted = series.shift(1)
    for w in rolling_windows:
        df[f"roll_mean_{w}"] = shifted.rolling(w).mean()
        df[f"roll_std_{w}"] = shifted.rolling(w).std()

    month = series.index.month
    df["month_sin"] = np.sin(2 * np.pi * month / 12.0)
    df["month_cos"] = np.cos(2 * np.pi * month / 12.0)
    df["trend"] = np.arange(len(series), dtype=float)

    return df.dropna()

feature_table = make_feature_table(y)
feature_table.head(3)

### Why use sin/cos for month?

Month is cyclical: December (12) and January (1) are adjacent in time. Encoding month as integer would incorrectly treat 12 and 1 as far apart. Sin/cos preserves circular geometry so nearby seasons stay nearby in feature space.

### The most common leakage mistake

Rolling features must use only prior observations. In this notebook we use:

-   `shifted = series.shift(1)`
-   then rolling statistics on `shifted`

This avoids using the current target value in its own predictors.

If we incorrectly used `series.rolling(w).mean()` at row $t$, then: $$ \tilde{\mu}_t^{(w)} = \frac{1}{w}\sum_{j=0}^{w-1} y_{t-j} $$ which includes $y_t$ itself. This leaks target information directly into predictors.

### Split the feature table by time

In [ ]:
train_df = feature_table.loc[:train_end]
val_df = feature_table.loc["1990-01-01":val_end]
test_df = feature_table.loc[test_start:]

X_train, y_train_tab = train_df.drop(columns="y"), train_df["y"]
X_val, y_val_tab = val_df.drop(columns="y"), val_df["y"]
X_test, y_test_tab = test_df.drop(columns="y"), test_df["y"]

print("Feature rows -> train:", len(X_train), "| val:", len(X_val), "| test:", len(X_test))

Pick one timestamp and manually verify that `lag_12` really corresponds to the value 12 months earlier. This simple manual check catches many indexing bugs early.

## Model comparison on validation

We compare a regularized linear model against a tree-based nonlinear model.

Why these two families:

-   **Ridge**: fast, stable baseline with interpretable linear effects.
-   **XGBoost / HistGBR**: captures nonlinear interactions and threshold behavior.

In many real projects, this pair already gives a strong "simple vs flexible" benchmark.

### Ridge regression objective

Ridge solves: $$ \hat{\beta} = \arg\min_{\beta} \left[ \frac{1}{n}\|y - X\beta\|_2^2 + \alpha \|\beta\|_2^2 \right] $$ Interpretation:

-   first term: fit training data,
-   second term: shrink coefficients toward zero to reduce variance.

The penalty is especially useful when lag features are correlated (common in time series).

### Gradient boosting objective (intuition)

Boosting builds an additive model: $$ f_M(x) = \sum_{m=1}^{M}\eta \, h_m(x) $$ where each weak learner $h_m$ is fit to current residual structure.

For squared error, each stage approximately learns what previous stages missed: $$ r_i^{(m)} \approx y_i - f_{m-1}(x_i) $$ This allows nonlinear interactions (for example, different behavior in different seasonal regimes).

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

def make_models():
    models = {
        "Ridge": Ridge(alpha=1.0),
    }

    if HAS_XGBOOST:
        models["XGBoost"] = XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
        )
    else:
        models["HistGBR"] = HistGradientBoostingRegressor(
            learning_rate=0.05,
            max_depth=4,
            random_state=42,
        )
    return models

models = make_models()
val_rows = []

for name, model in models.items():
    model.fit(X_train, y_train_tab)
    pred = model.predict(X_val)
    scores = forecast_metrics(y_val_tab, pred)
    val_rows.append({"Model": name, **scores})

val_results = pd.DataFrame(val_rows).set_index("Model").sort_values("MAE")
val_results.round(3)

**How to read validation results**:

-   If metrics are close, prefer the simpler model for maintainability.
-   If one model wins on MAE but loses badly on RMSE, inspect whether it occasionally makes very large errors.
-   A single split can still be noisy; backtesting below provides a more reliable estimate.

Also check rank stability across metrics. A model that wins only one metric by a tiny margin may not be practically better.

### Refit the best model and evaluate on test

In [ ]:
best_model_name = val_results.index[0]
print("Selected model:", best_model_name)

X_dev = pd.concat([X_train, X_val])
y_dev = pd.concat([y_train_tab, y_val_tab])

best_model = make_models()[best_model_name]
best_model.fit(X_dev, y_dev)
test_pred = best_model.predict(X_test)

test_results = pd.DataFrame({
    "Seasonal Naive (t-12)": forecast_metrics(y_test_tab, y.shift(12).loc[y_test_tab.index]),
    best_model_name: forecast_metrics(y_test_tab, test_pred),
}).T

test_results.round(3)

### Optional interpretation: feature influence

For teaching, it is useful to show what signals the selected model relies on.

In [ ]:
if best_model_name == "Ridge":
    coefs = pd.Series(best_model.coef_, index=X_dev.columns).sort_values(key=np.abs, ascending=False)
    print("Top Ridge coefficients (absolute value):")
    print(coefs.head(8).round(3))
elif best_model_name in ("XGBoost", "HistGBR"):
    if hasattr(best_model, "feature_importances_"):
        importances = pd.Series(best_model.feature_importances_, index=X_dev.columns).sort_values(ascending=False)
        print("Top feature importances:")
        print(importances.head(8).round(3))

### Plot forecast vs actual on test period

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(y_test_tab.index, y_test_tab.values, label="Actual", lw=2)
plt.plot(y_test_tab.index, y.shift(12).loc[y_test_tab.index].values, label="Seasonal Naive", ls="--")
plt.plot(y_test_tab.index, test_pred, label=best_model_name, lw=2)
plt.title("Test Forecast Comparison")
plt.ylabel("CO2 (ppm)")
plt.legend()
plt.show()

### Residual diagnostics (simple)

Good forecast plots can still hide systematic errors. We inspect residuals:

In [ ]:
residuals = y_test_tab.values - test_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(y_test_tab.index, residuals)
axes[0].axhline(0.0, color="black", ls="--", lw=1)
axes[0].set_title("Residuals Over Time")

axes[1].hist(residuals, bins=15, edgecolor="black")
axes[1].set_title("Residual Distribution")
plt.tight_layout()
plt.show()

print("Residual mean:", round(residuals.mean(), 4))
print("Residual std: ", round(residuals.std(), 4))

If residuals show clear temporal structure (e.g., long runs above zero), the model still misses dynamics.

### Residual autocorrelation check

For a well-specified one-step model, residual autocorrelation should be close to zero at small lags.

In [ ]:
for lag in [1, 2, 3, 6, 12]:
    ac = pd.Series(residuals).autocorr(lag=lag)
    print(f"Residual autocorr lag={lag:>2}: {ac:.3f}")

Persistent residual autocorrelation means "there is still predictable structure left in the errors."

## Walk-forward backtesting

Single splits can be noisy. We also do rolling-origin backtesting on train+val.

There are two common backtesting setups:

-   **Expanding window**: training set grows as time moves forward.
-   **Sliding window**: fixed-size recent history only.

`TimeSeriesSplit` behaves like expanding-window validation by default.

### Backtesting in mathematical form

Choose ordered cutoffs $\tau_1 < \tau_2 < \dots < \tau_K$. For fold $k$:

-   train on $\{1, \dots, \tau_k\}$,
-   test on $\{\tau_k + 1, \dots, \tau_k + h\}$.

Compute fold metric $M_k$, then aggregate: $$ \bar{M} = \frac{1}{K}\sum_{k=1}^{K} M_k $$ Optionally also report dispersion (std or IQR) across folds.

This estimates "expected future performance under repeated re-training" better than a single holdout.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

def walk_forward_backtest(model_factory, X, y, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    rows = []

    for fold, (tr_idx, te_idx) in enumerate(tscv.split(X), start=1):
        model = model_factory()
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

        model.fit(X_tr, y_tr)
        pred = model.predict(X_te)
        scores = forecast_metrics(y_te, pred)
        rows.append({"Fold": fold, **scores})

    return pd.DataFrame(rows)

cv_df = feature_table.loc[:val_end]
X_cv, y_cv = cv_df.drop(columns="y"), cv_df["y"]

factories = {"Ridge": lambda: Ridge(alpha=1.0)}
if HAS_XGBOOST:
    factories["XGBoost"] = lambda: XGBRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
    )
else:
    factories["HistGBR"] = lambda: HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=4,
        random_state=42,
    )

for name, fn in factories.items():
    bt = walk_forward_backtest(fn, X_cv, y_cv, n_splits=5)
    print(f"\n{name}")
    print(bt.round(3))
    print("Mean:", bt[["MAE", "RMSE", "MAPE(%)"]].mean().round(3).to_dict())

**How to use backtest output in practice**:

-   Report both mean and fold-to-fold variability.
-   If one fold is much worse, inspect that calendar period for anomalies or regime changes.
-   Select models that are not only accurate on average, but also stable across folds.

## Practical takeaways

1.  Start with **seasonal naive** before complex models.
2.  Build lag/rolling/calendar features using only historical information.
3.  Use date-based splits and walk-forward validation.
4.  Keep a robust baseline in production monitoring.
5.  Combine performance with interpretability and stability when choosing the final model.

## Exercises

### Leakage trap (random split)

Run a random train/test split and compare to proper time split. Why is random splitting over-optimistic?

Which assumptions of forecasting are violated by random shuffling?

### Direct vs recursive multi-step forecasting

Implement a 6-step forecast:

-   **Recursive**: one-step model called repeatedly.
-   **Direct**: separate model per horizon.

Recursive forecasts accumulate error; direct forecasts can reduce accumulation but require more models.

### Add domain-specific covariates

Add a new exogenous feature (for example, policy period indicator, temperature proxy, or instrument change marker) and compare:

-   original model predictions (without the covariate)
-   augmented model predictions (with the covariate)

Keep model family, hyperparameters, and split fixed so the comparison is fair.

Do the augmented predictions improve MAE/RMSE/MAPE consistently, or only one metric? If improvements are small or unstable, could the covariate be spurious?

## Summary

-   Time-series ML is mostly about *correct framing* and *leakage control*.
-   Feature-based models are strong, transparent, and often hard to beat on medium-size datasets.
-   Backtesting is essential for believable error estimates.
-   Explanation quality matters: each modeling step should be defensible in terms of what would be known at prediction time.